# Smart Planner & Burnout Prevention System - Database Data Generator

This notebook generates test data for the database, inserting:
- **2,500 Users** (Name format: `Generated User [i]`, Email: `generateduser[i]@[random domain]`, password: `PW!Generated_User_[i]`)
- **12,500 Tasks** (5 tasks per user, randomly categorized, prioritized, and set to Done/Undone status)
- **12,500 Study Sessions** (5 focus sessions per user, optionally linked to generated tasks)
- **12,500 Burnout Logs** (5 daily wellness logs per user, calculating index dynamically using the backend scoring algorithm)

We will load environment variables from the `backend/.env` file.

In [1]:
# 1. Install required packages
!pip install mysql-connector-python python-dotenv

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 2. Imports and Environment Setup
import os
import random
import datetime
from dotenv import load_dotenv
import mysql.connector

# Load database settings from backend/.env
env_path = os.path.abspath(os.path.join("..", "..", "backend", ".env"))
print(f"Loading environment variables from: {env_path}")
load_dotenv(dotenv_path=env_path)

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT", 3306)

print(f"Host: {db_host}")
print(f"User: {db_user}")
print(f"Database: {db_name}")
print(f"Port: {db_port}")

Loading environment variables from: d:\HCI-Final_Project\backend\.env
Host: mysql-3e35218b-kst-rithh07.e.aivencloud.com
User: avnadmin
Database: Mindful_Study
Port: 22859


In [3]:
# 3. Establish Connection & Retrieve Metadata
conn = mysql.connector.connect(
    host=db_host,
    user=db_user,
    password=db_password,
    database=db_name,
    port=int(db_port)
)
cursor = conn.cursor()

# Retrieve categories list to dynamically link to tasks
cursor.execute("SELECT category_id, type FROM Categories")
categories = cursor.fetchall()

print(f"Available Categories: {categories}")
category_ids = [c[0] for c in categories]
if not category_ids:
    print("WARNING: No categories found in the database. Please ensure they are loaded.")

Available Categories: [(1, 'Practice'), (2, 'Assignment'), (3, 'Project'), (4, 'Revision'), (5, 'Research'), (6, 'Others')]


In [4]:
# 4. Define Helper Functions & Rules for Burnout Index calculation
# This matches the scoring logic implemented in backend's analyticsController.js

def calculate_burnout_index(mood, sleep_hrs, sleep_q, screen, pending, overdue, missed):
    mood_map = {
        'Happy': 20,
        'Normal': 40,
        'Tired': 60,
        'Frustrated': 80,
        'Stressed': 100
    }
    mood_score = mood_map.get(mood, 60)

    sleep_map = {
        'Under 4 hours': 100,
        '5-6 hours': 75,
        '7-8 hours': 25,
        'Above 8 hours': 50
    }
    sleep_score = sleep_map.get(sleep_hrs, 25)

    quality_map = {
        1: 100, 2: 80, 3: 60, 4: 40, 5: 20
    }
    sleep_quality_score = quality_map.get(int(sleep_q), 60)

    screen_map = {
        'Under 4 hours': 25,
        '5-6 hours': 50,
        '7-8 hours': 75,
        'Above 8 hours': 100
    }
    screen_time_score = screen_map.get(screen, 50)

    pending_score = 0
    if pending >= 5:
        pending_score = 100
    elif pending >= 3:
        pending_score = 65
    elif pending >= 1:
        pending_score = 30

    overdue_score = 0
    if overdue >= 2:
        overdue_score = 100
    elif overdue == 1:
        overdue_score = 50

    missed_score = 0
    if missed >= 2:
        missed_score = 100
    elif missed == 1:
        missed_score = 50

    index_value = round(
        mood_score * 0.3 +
        sleep_score * 0.1 +
        sleep_quality_score * 0.1 +
        screen_time_score * 0.1 +
        pending_score * 0.2 +
        overdue_score * 0.1 +
        missed_score * 0.1
    )
    return index_value

In [5]:
# 5. Seed Insertion Script (Batch execution: 500 users at a time)
num_users = 2500
batch_size = 500
email_domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'edu.com']

genders = ['Male', 'Female', 'Other', 'Prefer not to say']
moods = ['Happy', 'Normal', 'Tired', 'Frustrated', 'Stressed']
sleep_options = ['Under 4 hours', '5-6 hours', '7-8 hours', 'Above 8 hours']
screen_options = ['Under 4 hours', '5-6 hours', '7-8 hours', 'Above 8 hours']
priorities = ['High', 'Medium', 'Low']
statuses = ['Done', 'Undone']
techniques = ['Pomodoro', '50/10 Rule', 'Short Focus', 'Custom']
breaks = ['5 min', '10 min', '15 min']

total_users_inserted = 0
total_tasks_inserted = 0
total_sessions_inserted = 0
total_burnout_inserted = 0

print(f"Starting seed data insertion for {num_users} users...")

for start_idx in range(1, num_users + 1, batch_size):
    end_idx = min(start_idx + batch_size - 1, num_users)
    print(f"Processing users {start_idx} to {end_idx}...")
    
    users_batch = []
    for i in range(start_idx, end_idx + 1):
        full_name = f"Generated User {i}"
        domain = random.choice(email_domains)
        email = f"generateduser{i}@{domain}"
        password = f"PW!Generated_User_{i}"
        age = random.randint(18, 45)
        gender = random.choice(genders)
        
        created_days_ago = random.randint(10, 60)
        created_at_date = datetime.date.today() - datetime.timedelta(days=created_days_ago)
        
        dob = datetime.date.today() - datetime.timedelta(days=age * 365 + random.randint(0, 364))
        mode_pref = random.choice(['Light', 'Dark'])
        
        users_batch.append((full_name, email, password, age, gender, dob, created_at_date, False, mode_pref))
    
    # Bulk insert users
    user_insert_query = """
        INSERT INTO Users (full_name, email, password, age, gender, dob, created_at, isNewUser, modePreference)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(user_insert_query, users_batch)
    conn.commit()
    
    # Fetch inserted user_ids
    batch_emails = [u[1] for u in users_batch]
    format_strings = ','.join(['%s'] * len(batch_emails))
    cursor.execute(f"SELECT user_id, created_at FROM Users WHERE email IN ({format_strings})", batch_emails)
    user_records = cursor.fetchall()
    
    tasks_batch = []
    
    # 1. Generate tasks first in a batch
    for user_id, user_created_date in user_records:
        for j in range(1, 6):
            title = f"Task {j} for User {user_id}"
            description = f"This is the description for task {j} of generated user {user_id}"
            priority = random.choice(priorities)
            status = random.choice(statuses)
            category_id = random.choice(category_ids) if category_ids else None
            
            task_created_days = random.randint(0, 5)
            task_created_date = user_created_date + datetime.timedelta(days=task_created_days)
            due_days = random.randint(1, 14)
            due_date = task_created_date + datetime.timedelta(days=due_days)
            
            completed_at = None
            if status == 'Done':
                completed_at_days = random.randint(0, due_days)
                completed_at = datetime.datetime.combine(task_created_date + datetime.timedelta(days=completed_at_days), datetime.time(random.randint(9, 18), random.randint(0, 59)))
            
            tasks_batch.append((user_id, category_id, title, description, priority, status, due_date, task_created_date, completed_at))
            
    task_insert_query = """
        INSERT INTO Tasks (user_id, category_id, title, description, priority, status, due_date, created_at, completed_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(task_insert_query, tasks_batch)
    conn.commit()
    
    # Retrieve the inserted tasks to link sessions
    batch_user_ids = [u[0] for u in user_records]
    format_strings_users = ','.join(['%s'] * len(batch_user_ids))
    cursor.execute(f"SELECT task_id, user_id, created_at, status, due_date FROM Tasks WHERE user_id IN ({format_strings_users})", batch_user_ids)
    task_records = cursor.fetchall()
    
    # Map tasks by user_id
    user_tasks_map = {}
    for task_id, u_id, t_created, t_status, t_due in task_records:
        if u_id not in user_tasks_map:
            user_tasks_map[u_id] = []
        user_tasks_map[u_id].append({'task_id': task_id, 'created_at': t_created, 'status': t_status, 'due_date': t_due})
        
    sessions_batch = []
    burnout_batch = []
    
    # 2. Generate study sessions and burnout logs
    for user_id, user_created_date in user_records:
        u_tasks = user_tasks_map.get(user_id, [])
        
        pending_count = sum(1 for t in u_tasks if t['status'] == 'Undone')
        overdue_count = sum(1 for t in u_tasks if t['status'] == 'Undone' and t['due_date'] < datetime.date.today())
        
        u_sessions = []
        for k in range(1, 6):
            task_choice = random.choice(u_tasks) if u_tasks else None
            task_id = task_choice['task_id'] if task_choice else None
            
            duration = random.choice([15, 25, 50, 60])
            start_days = random.randint(0, 10)
            start_date = user_created_date + datetime.timedelta(days=start_days)
            start_time = datetime.datetime.combine(start_date, datetime.time(random.randint(8, 20), random.randint(0, 59)))
            end_time = start_time + datetime.timedelta(minutes=duration)
            
            title = f"Study Session {k} for User {user_id}"
            technique = random.choice(techniques)
            brk = random.choice(breaks)
            burnout_prev = random.choice([0, 1])
            is_completed = random.choice([0, 1])
            
            u_sessions.append({'start_time': start_time, 'is_completed': is_completed})
            sessions_batch.append((user_id, task_id, start_time, end_time, duration, title, technique, brk, burnout_prev, is_completed))
            
        missed_count = sum(1 for s in u_sessions if s['start_time'] < datetime.datetime.now() and s['is_completed'] == 0)
        
        for m in range(1, 6):
            mood = random.choice(moods)
            sleep_h = random.choice(sleep_options)
            sleep_q = random.randint(1, 5)
            screen = random.choice(screen_options)
            note = f"Check-in log {m} for User {user_id}"
            
            burnout_idx = calculate_burnout_index(mood, sleep_h, sleep_q, screen, pending_count, overdue_count, missed_count)
            
            log_days = random.randint(1, 12)
            log_date = user_created_date + datetime.timedelta(days=log_days)
            
            burnout_batch.append((user_id, mood, sleep_h, sleep_q, screen, note, burnout_idx, log_date))

    # Bulk insert study sessions
    session_insert_query = """
        INSERT INTO StudySessions (user_id, task_id, start_time, end_time, duration_minutes, title, focus_technique, break_duration, burnout_prevention, is_completed)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(session_insert_query, sessions_batch)
    
    # Bulk insert burnout logs
    burnout_insert_query = """
        INSERT INTO BurnoutLogs (user_id, mood_level, sleep_hours, sleep_quality, screen_time, note, burnout_index, created_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(burnout_insert_query, burnout_batch)
    
    conn.commit()
    
    total_users_inserted += len(users_batch)
    total_tasks_inserted += len(tasks_batch)
    total_sessions_inserted += len(sessions_batch)
    total_burnout_inserted += len(burnout_batch)

print("Insertion process complete!")
print(f"Total Users Inserted: {total_users_inserted}")
print(f"Total Tasks Inserted: {total_tasks_inserted}")
print(f"Total Study Sessions Inserted: {total_sessions_inserted}")
print(f"Total Burnout Logs Inserted: {total_burnout_inserted}")

cursor.close()
conn.close()

Starting seed data insertion for 2500 users...
Processing users 1 to 500...
Processing users 501 to 1000...
Processing users 1001 to 1500...
Processing users 1501 to 2000...
Processing users 2001 to 2500...
Insertion process complete!
Total Users Inserted: 2500
Total Tasks Inserted: 12500
Total Study Sessions Inserted: 12500
Total Burnout Logs Inserted: 12500
